# BRFSS Feature Availability Verifier

Notebook này check **xem feature nào trong list candidate có raw column resolve được ở từng năm BRFSS 2019/2020/2021**.

Output sẽ phân loại mỗi feature thành:
- **Always (3/3)** — có ở mọi năm → thêm thẳng vào baseline (không thuộc experiment fill-missing).
- **Rotating (1–2 / 3)** — thiếu ở ≥1 năm → **candidate cho fill-missing experiment**.
- **Never (0/3)** — không có ở năm nào → loại bỏ.

Có 2 cấp độ check:
1. **Header check** (siêu nhanh): chỉ đọc tên cột.
2. **Signal check** (chậm hơn ~3 phút): load 20K rows mẫu, tính tỉ lệ non-null + valid (loại sentinel).

Có thể chạy trên Colab với data trong Google Drive.

## 1. Mount Google Drive (chỉ Colab)

In [ ]:
# Skip cell này nếu đang chạy local (không phải Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not on Colab — skip mounting drive.")

Mounted at /content/drive


## 2. CONFIG (paths)

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/dataset_brfss")

CONFIG = {
    "y2019": str(BASE_DIR / "LLCP2019_CSV.csv"),
    "y2020": str(BASE_DIR / "LLCP2020_CSV.csv"),
    "y2021": str(BASE_DIR / "LLCP2021_CSV.csv"),
    # Signal check: load N rows mẫu để tính tỉ lệ valid. None = chỉ header check.
    "sample_rows_for_signal": 20000,
    # Output
    "outdir": "./feature_availability_outputs",
}

# Validate paths
for k in ["y2019", "y2020", "y2021"]:
    p = Path(CONFIG[k])
    print(f"{k}: {p}  ->  {'EXISTS' if p.exists() else 'MISSING'}")


y2019: /content/drive/MyDrive/dataset_brfss/LLCP2019_CSV.csv  ->  EXISTS
y2020: /content/drive/MyDrive/dataset_brfss/LLCP2020_CSV.csv  ->  EXISTS
y2021: /content/drive/MyDrive/dataset_brfss/LLCP2021_CSV.csv  ->  EXISTS


## 3. Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional

SPECIAL_MISSING_GENERIC = {7, 8, 9, 77, 88, 99, 777, 888, 999, 7777, 8888, 9999, 99900}

OUTDIR = Path(CONFIG["outdir"])
OUTDIR.mkdir(parents=True, exist_ok=True)


## 4. Danh sách candidates

25 features được nhóm theo 4 tier:

- **Tier 0**: 18 features baseline hiện tại (sanity check — kỳ vọng all 3/3).
- **Tier 1**: top-3 CVD predictors (HighBP, HighChol, Stroke) + cholesterol check + functional disability.
- **Tier 2**: SES (Income, Education, Insurance, Marital, Employment).
- **Tier 3**: Lifestyle (Sleep, Mental/Physical health days, Aspirin, E-cigarette, drinking).
- **Tier 4**: Demographic stratifiers (Race, Hispanic).

Mỗi canonical feature có nhiều raw column candidate để chịu được rotating tên cột giữa các năm.

In [ ]:
CANDIDATES: Dict[str, List[str]] = {
    # ===== Tier 0: Existing baseline 18 features =====
    'General_Health':              ['GENHLTH'],
    'Checkup':                     ['CHECKUP1'],
    'Exercise':                    ['EXERANY2'],
    'Skin_Cancer':                 ['CHCSCNCR'],
    'Other_Cancer':                ['CHCOCNCR'],
    'Depression':                  ['ADDEPEV3'],
    'Diabetes':                    ['DIABETE4', 'DIABETE3'],
    'Arthritis':                   ['HAVARTH5', 'HAVARTH4'],
    'Sex':                         ['SEXVAR', '_SEX', 'BIRTHSEX'],
    'Age_Category':                ['_AGEG5YR', '_AGE65YR', '_AGE_G'],
    'Height_(cm)':                 ['HTM4', 'HTIN4', 'HEIGHT3'],
    'Weight_(kg)':                 ['WTKG3', 'WEIGHT2'],
    'BMI':                         ['_BMI5', 'BMI'],
    'Smoking_History':             ['SMOKE100'],
    'Alcohol_Consumption':         ['_DRNKWK1', 'AVEDRNK3', 'DRNKANY5'],
    'Fruit_Consumption':           ['FRUIT2', 'FRUIT1'],
    'Green_Vegetables_Consumption':['FVGREEN1', 'FVGREEN'],
    'FriedPotato_Consumption':     ['FRENCHF1', 'FRENCHF'],

    # ===== Tier 1: Top CVD predictors =====
    'High_BP':                     ['_RFHYPE5', '_RFHYPE6', 'BPHIGH4', 'BPHIGH6'],
    'High_Chol':                   ['_RFCHOL3', '_RFCHOL2', 'TOLDHI3', 'TOLDHI2'],
    'Cholesterol_Checked':         ['_CHOLCH3', '_CHOLCHK', 'CHOLCHK3', 'CHOLCHK2'],
    'Stroke':                      ['CVDSTRK3'],
    'Difficulty_Walking':          ['DIFFWALK'],
    'Difficulty_Dressing':         ['DIFFDRES'],
    'Difficulty_Errands':          ['DIFFALON'],

    # ===== Tier 2: SES =====
    'Income':                      ['INCOME3', 'INCOME2'],
    'Education':                   ['EDUCA'],
    'Marital_Status':              ['MARITAL'],
    'Employment':                  ['EMPLOY1'],
    'Health_Insurance':            ['_HLTHPLN', 'HLTHPLN1', 'HLTHPL1'],

    # ===== Tier 3: Lifestyle / behavior =====
    'Sleep_Hours':                 ['SLEPTIM1'],
    'Mental_Health_Days':          ['MENTHLTH'],
    'Physical_Health_Days':        ['PHYSHLTH'],
    'Poor_Health_Days':            ['POORHLTH'],
    'Aspirin_Use':                 ['ASPIRIN', 'CVDASPRN', 'ASPUNSAF'],
    'E_Cigarette':                 ['ECIGNOW1', 'ECIGNOW', 'ECIGARET'],
    'Heavy_Drinking':              ['_RFDRHV7', '_RFDRHV6'],
    'Binge_Drinking':              ['_RFBING5', '_RFBING6'],

    # ===== Tier 4: Demographics =====
    'Race':                        ['_IMPRACE', '_RACE', '_RACE_G1'],
    'Hispanic':                    ['_HISPANC'],
    'Urban_Rural':                 ['_URBSTAT', 'MSCODE'],
}

# Tier label mapping (purely metadata for output)
TIER_OF = {}
_t0 = ['General_Health','Checkup','Exercise','Skin_Cancer','Other_Cancer','Depression',
       'Diabetes','Arthritis','Sex','Age_Category','Height_(cm)','Weight_(kg)','BMI',
       'Smoking_History','Alcohol_Consumption','Fruit_Consumption',
       'Green_Vegetables_Consumption','FriedPotato_Consumption']
_t1 = ['High_BP','High_Chol','Cholesterol_Checked','Stroke','Difficulty_Walking',
       'Difficulty_Dressing','Difficulty_Errands']
_t2 = ['Income','Education','Marital_Status','Employment','Health_Insurance']
_t3 = ['Sleep_Hours','Mental_Health_Days','Physical_Health_Days','Poor_Health_Days',
       'Aspirin_Use','E_Cigarette','Heavy_Drinking','Binge_Drinking']
_t4 = ['Race','Hispanic','Urban_Rural']
for f in _t0: TIER_OF[f] = 0
for f in _t1: TIER_OF[f] = 1
for f in _t2: TIER_OF[f] = 2
for f in _t3: TIER_OF[f] = 3
for f in _t4: TIER_OF[f] = 4

print(f"Total candidates: {len(CANDIDATES)}")
print(f"  Tier 0 (baseline existing): {len(_t0)}")
print(f"  Tier 1 (top CVD predictors): {len(_t1)}")
print(f"  Tier 2 (SES):                {len(_t2)}")
print(f"  Tier 3 (lifestyle):          {len(_t3)}")
print(f"  Tier 4 (demographics):       {len(_t4)}")


Total candidates: 41
  Tier 0 (baseline existing): 18
  Tier 1 (top CVD predictors): 7
  Tier 2 (SES):                5
  Tier 3 (lifestyle):          8
  Tier 4 (demographics):       3


## 5. Header check (siêu nhanh, chỉ đọc tên cột)

In [ ]:
def read_header(path: str) -> set:
    cols = pd.read_csv(path, nrows=0, low_memory=False).columns.tolist()
    return set(c for c in cols if c != 'Unnamed: 0')

def first_existing(columns: set, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in columns:
            return c
    return None

print("Reading headers...")
year_to_cols: Dict[int, set] = {}
for year in [2019, 2020, 2021]:
    cols = read_header(CONFIG[f"y{year}"])
    year_to_cols[year] = cols
    print(f"  {year}: {len(cols)} columns")

# Per-year resolve of each canonical feature
resolved_table = []
for feat, cands in CANDIDATES.items():
    row = {"feature": feat, "tier": TIER_OF.get(feat, "?")}
    for year in [2019, 2020, 2021]:
        row[f"raw_{year}"] = first_existing(year_to_cols[year], cands)
        row[f"has_{year}"] = int(row[f"raw_{year}"] is not None)
    row["n_years"] = row["has_2019"] + row["has_2020"] + row["has_2021"]
    resolved_table.append(row)

resolved_df = pd.DataFrame(resolved_table)
resolved_df = resolved_df.sort_values(["tier", "n_years", "feature"], ascending=[True, False, True]).reset_index(drop=True)
print(f"\nTotal candidates checked: {len(resolved_df)}")
display(resolved_df)


Reading headers...
  2019: 342 columns
  2020: 279 columns
  2021: 303 columns

Total candidates checked: 41


,feature,tier,raw_2019,has_2019,raw_2020,has_2020,raw_2021,has_2021,n_years
0,Age_Category,0,_AGEG5YR,1,_AGEG5YR,1,_AGEG5YR,1,3
1,Alcohol_Consumption,0,_DRNKWK1,1,_DRNKWK1,1,_DRNKWK1,1,3
2,Arthritis,0,HAVARTH4,1,HAVARTH4,1,HAVARTH5,1,3
3,BMI,0,_BMI5,1,_BMI5,1,_BMI5,1,3
4,Checkup,0,CHECKUP1,1,CHECKUP1,1,CHECKUP1,1,3
5,Depression,0,ADDEPEV3,1,ADDEPEV3,1,ADDEPEV3,1,3
6,Diabetes,0,DIABETE4,1,DIABETE4,1,DIABETE4,1,3
7,Exercise,0,EXERANY2,1,EXERANY2,1,EXERANY2,1,3
8,General_Health,0,GENHLTH,1,GENHLTH,1,GENHLTH,1,3
9,Height_(cm),0,HTM4,1,HTM4,1,HTM4,1,3


## 6. Phân loại Always / Rotating / Never

In [ ]:
ALWAYS = resolved_df[resolved_df["n_years"] == 3].copy()
ROTATING = resolved_df[(resolved_df["n_years"] >= 1) & (resolved_df["n_years"] <= 2)].copy()
NEVER = resolved_df[resolved_df["n_years"] == 0].copy()

print("="*70)
print(f"ALWAYS (3/3 years) — kỳ vọng baseline đã có hoặc thêm vào baseline thẳng")
print(f"  Count: {len(ALWAYS)}")
print("="*70)
display(ALWAYS[["feature","tier","raw_2019","raw_2020","raw_2021"]])

print("\n" + "="*70)
print(f"ROTATING (1–2 years) — CANDIDATE cho fill-missing experiment")
print(f"  Count: {len(ROTATING)}")
print("="*70)
display(ROTATING[["feature","tier","raw_2019","raw_2020","raw_2021","n_years"]])

print("\n" + "="*70)
print(f"NEVER (0/3 years) — không có raw column ở năm nào, loại bỏ")
print(f"  Count: {len(NEVER)}")
print("="*70)
display(NEVER[["feature","tier"]])


ALWAYS (3/3 years) — kỳ vọng baseline đã có hoặc thêm vào baseline thẳng
  Count: 32


,feature,tier,raw_2019,raw_2020,raw_2021
0,Age_Category,0,_AGEG5YR,_AGEG5YR,_AGEG5YR
1,Alcohol_Consumption,0,_DRNKWK1,_DRNKWK1,_DRNKWK1
2,Arthritis,0,HAVARTH4,HAVARTH4,HAVARTH5
3,BMI,0,_BMI5,_BMI5,_BMI5
4,Checkup,0,CHECKUP1,CHECKUP1,CHECKUP1
5,Depression,0,ADDEPEV3,ADDEPEV3,ADDEPEV3
6,Diabetes,0,DIABETE4,DIABETE4,DIABETE4
7,Exercise,0,EXERANY2,EXERANY2,EXERANY2
8,General_Health,0,GENHLTH,GENHLTH,GENHLTH
9,Height_(cm),0,HTM4,HTM4,HTM4



ROTATING (1–2 years) — CANDIDATE cho fill-missing experiment
  Count: 9


,feature,tier,raw_2019,raw_2020,raw_2021,n_years
15,FriedPotato_Consumption,0,FRENCHF1,None,FRENCHF1,2
16,Fruit_Consumption,0,FRUIT2,None,FRUIT2,2
17,Green_Vegetables_Consumption,0,FVGREEN1,None,FVGREEN1,2
22,Cholesterol_Checked,1,CHOLCHK2,None,_CHOLCH3,2
23,High_BP,1,_RFHYPE5,None,_RFHYPE6,2
24,High_Chol,1,_RFCHOL2,None,_RFCHOL3,2
35,E_Cigarette,3,None,ECIGNOW,ECIGNOW1,2
36,Aspirin_Use,3,ASPIRIN,None,None,1
37,Sleep_Hours,3,None,SLEPTIM1,None,1



NEVER (0/3 years) — không có raw column ở năm nào, loại bỏ
  Count: 0


,feature,tier


## 7. Signal check (load sample 20K rows / năm)

Với mỗi feature đã resolve, tính:
- **% non-null** (cột không thiếu thuần)
- **% valid** (loại special missing sentinels: 7, 9, 77, 99, 777, 999, ...)

Feature có **% valid < 50%** ở năm nào → không đủ signal để impute / dùng làm feature.

Cell này chậm hơn (~2–3 phút). Có thể bỏ qua nếu chỉ cần header check.

In [ ]:
sample_n = CONFIG.get("sample_rows_for_signal")

if sample_n is None:
    print("sample_rows_for_signal=None — bỏ qua signal check.")
    signal_df = None
else:
    print(f"Loading {sample_n} sample rows per year for signal check...")
    year_to_sample: Dict[int, pd.DataFrame] = {}
    for year in [2019, 2020, 2021]:
        # chỉ load các raw columns cần
        all_cands = set()
        for cands in CANDIDATES.values():
            all_cands.update(cands)
        cols_in_file = year_to_cols[year]
        usecols = [c for c in cols_in_file if c in all_cands]
        df_y = pd.read_csv(CONFIG[f"y{year}"], usecols=usecols, nrows=sample_n, low_memory=False)
        year_to_sample[year] = df_y
        print(f"  {year}: loaded shape {df_y.shape}")

    # Compute validity per feature per year
    rows = []
    for _, row in resolved_df.iterrows():
        feat = row["feature"]
        rec = {"feature": feat, "tier": row["tier"], "n_years": row["n_years"]}
        for year in [2019, 2020, 2021]:
            raw = row[f"raw_{year}"]
            if raw is None:
                rec[f"nonnull_{year}"] = None
                rec[f"valid_{year}"] = None
                continue
            s = pd.to_numeric(year_to_sample[year][raw], errors='coerce')
            n_total = len(s)
            n_nonnull = int(s.notna().sum())
            n_invalid_sentinel = int(s.isin(SPECIAL_MISSING_GENERIC).sum())
            n_valid = n_nonnull - n_invalid_sentinel
            rec[f"nonnull_{year}"] = round(n_nonnull / n_total, 3) if n_total else None
            rec[f"valid_{year}"]   = round(n_valid   / n_total, 3) if n_total else None
        rows.append(rec)

    signal_df = pd.DataFrame(rows)
    signal_df = signal_df.sort_values(["tier", "n_years", "feature"], ascending=[True, False, True]).reset_index(drop=True)
    display(signal_df)


Loading 20000 sample rows per year for signal check...
  2019: loaded shape (20000, 53)
  2020: loaded shape (20000, 46)
  2021: loaded shape (20000, 53)


,feature,tier,n_years,nonnull_2019,valid_2019,nonnull_2020,valid_2020,nonnull_2021,valid_2021
0,Age_Category,0,3,1.000,0.727,1.000,0.735,1.000,0.732
1,Alcohol_Consumption,0,3,1.000,0.931,1.000,0.921,1.000,0.930
2,Arthritis,0,3,1.000,0.995,1.000,0.995,1.000,0.995
3,BMI,0,3,0.919,0.919,0.905,0.905,0.913,0.913
4,Checkup,0,3,1.000,0.984,1.000,0.980,1.000,0.980
5,Depression,0,3,1.000,0.995,1.000,0.994,1.000,0.994
6,Diabetes,0,3,1.000,0.998,1.000,0.998,1.000,0.998
7,Exercise,0,3,0.946,0.944,1.000,0.998,1.000,0.998
8,General_Health,0,3,1.000,0.997,1.000,0.997,1.000,0.997
9,Height_(cm),0,3,0.959,0.959,0.947,0.947,0.954,0.954


## 8. Khuyến nghị cuối cùng

Dựa trên availability + signal:

- **`add_to_baseline`**: 3/3 years AND valid_rate ≥ 0.5 ở mọi năm → thêm vào baseline.
- **`fill_missing_candidate`**: 1–2/3 years AND valid_rate ≥ 0.5 ở năm có data → DÙNG cho experiment.
- **`drop`**: phần còn lại.

In [10]:
def classify_row(row, signal_df):
    n_years = row["n_years"]
    if n_years == 0:
        return "drop", "no raw column"

    # signal check
    if signal_df is not None:
        s_row = signal_df[signal_df["feature"] == row["feature"]].iloc[0]
        valid_rates = []
        for year in [2019, 2020, 2021]:
            v = s_row[f"valid_{year}"]
            if v is not None and not pd.isna(v):
                valid_rates.append(v)
        if not valid_rates:
            return "drop", "no valid data"
        if min(valid_rates) < 0.5:
            return "drop", f"low valid rate ({min(valid_rates):.2f})"

    if n_years == 3:
        return "add_to_baseline", "available all 3 years"
    return "fill_missing_candidate", f"available {n_years}/3 years"

decisions = []
for _, row in resolved_df.iterrows():
    decision, reason = classify_row(row, signal_df)
    decisions.append({
        "feature": row["feature"],
        "tier": row["tier"],
        "n_years": row["n_years"],
        "decision": decision,
        "reason": reason,
        "raw_2019": row["raw_2019"],
        "raw_2020": row["raw_2020"],
        "raw_2021": row["raw_2021"],
    })

decision_df = pd.DataFrame(decisions).sort_values(
    ["decision", "tier", "feature"], ascending=[True, True, True]
).reset_index(drop=True)

print("="*70)
print("FINAL RECOMMENDATIONS")
print("="*70)
for dec in ["add_to_baseline", "fill_missing_candidate", "drop"]:
    sub = decision_df[decision_df["decision"] == dec]
    print(f"\n>>> {dec.upper()}: {len(sub)} features")
    for _, r in sub.iterrows():
        print(f"   [tier {r['tier']}] {r['feature']:<28} "
              f"({r['n_years']}/3 years) — {r['reason']}")

# Save outputs
resolved_df.to_csv(OUTDIR / "header_check.csv", index=False)
if signal_df is not None:
    signal_df.to_csv(OUTDIR / "signal_check.csv", index=False)
decision_df.to_csv(OUTDIR / "final_decisions.csv", index=False)
print(f"\nSaved CSVs to: {OUTDIR.resolve()}")


FINAL RECOMMENDATIONS

>>> ADD_TO_BASELINE: 28 features
   [tier 0] Age_Category                 (3/3 years) — available all 3 years
   [tier 0] Alcohol_Consumption          (3/3 years) — available all 3 years
   [tier 0] Arthritis                    (3/3 years) — available all 3 years
   [tier 0] BMI                          (3/3 years) — available all 3 years
   [tier 0] Checkup                      (3/3 years) — available all 3 years
   [tier 0] Depression                   (3/3 years) — available all 3 years
   [tier 0] Diabetes                     (3/3 years) — available all 3 years
   [tier 0] Exercise                     (3/3 years) — available all 3 years
   [tier 0] General_Health               (3/3 years) — available all 3 years
   [tier 0] Height_(cm)                  (3/3 years) — available all 3 years
   [tier 0] Other_Cancer                 (3/3 years) — available all 3 years
   [tier 0] Sex                          (3/3 years) — available all 3 years
   [tier 0] Skin_Can

## 9. Snippet code để patch baseline notebook

Sau khi đã có `decision_df`, lấy ra:
- `add_to_baseline` features → thêm vào `FEATURE_CANDIDATES` của baseline notebook (intersection sẽ tự nhận).
- `fill_missing_candidate` features → dùng cho experiment fill-missing.

In ra Python dict ngay để bạn copy-paste sang notebook chính.

In [11]:
def emit_dict_block(features_subset, source_df=resolved_df):
    print("FEATURE_CANDIDATES_EXTRA = {")
    for feat in features_subset:
        cands = CANDIDATES[feat]
        cands_str = ", ".join(f"'{c}'" for c in cands)
        print(f"    '{feat}': [{cands_str}],")
    print("}")

print(">>> ADD TO BASELINE (intersection sẽ giữ ở mọi năm):")
add_features = decision_df[decision_df["decision"] == "add_to_baseline"]["feature"].tolist()
add_features = [f for f in add_features if TIER_OF.get(f, 0) > 0]  # bỏ tier 0 đã có sẵn
emit_dict_block(add_features)

print("\n>>> FILL MISSING CANDIDATES (dùng cho experiment KNN/MICE/AE/Statistical):")
fm_features = decision_df[decision_df["decision"] == "fill_missing_candidate"]["feature"].tolist()
emit_dict_block(fm_features)

print(f"\nTổng kết:")
print(f"  - {len(add_features)} features thêm vào baseline")
print(f"  - {len(fm_features)} features cho fill-missing experiment")

>>> ADD TO BASELINE (intersection sẽ giữ ở mọi năm):
FEATURE_CANDIDATES_EXTRA = {
    'Difficulty_Dressing': ['DIFFDRES'],
    'Difficulty_Errands': ['DIFFALON'],
    'Difficulty_Walking': ['DIFFWALK'],
    'Stroke': ['CVDSTRK3'],
    'Education': ['EDUCA'],
    'Employment': ['EMPLOY1'],
    'Health_Insurance': ['_HLTHPLN', 'HLTHPLN1', 'HLTHPL1'],
    'Marital_Status': ['MARITAL'],
    'Binge_Drinking': ['_RFBING5', '_RFBING6'],
    'Heavy_Drinking': ['_RFDRHV7', '_RFDRHV6'],
    'Hispanic': ['_HISPANC'],
    'Race': ['_IMPRACE', '_RACE', '_RACE_G1'],
    'Urban_Rural': ['_URBSTAT', 'MSCODE'],
}

>>> FILL MISSING CANDIDATES (dùng cho experiment KNN/MICE/AE/Statistical):
FEATURE_CANDIDATES_EXTRA = {
    'FriedPotato_Consumption': ['FRENCHF1', 'FRENCHF'],
    'Fruit_Consumption': ['FRUIT2', 'FRUIT1'],
    'Green_Vegetables_Consumption': ['FVGREEN1', 'FVGREEN'],
    'Cholesterol_Checked': ['_CHOLCH3', '_CHOLCHK', 'CHOLCHK3', 'CHOLCHK2'],
    'High_BP': ['_RFHYPE5', '_RFHYPE6', 'BPHIGH4',